# Semantic search, with Candle

A small pretrained model (sentence-transformers' all-MiniLM-L6-v2) turns short help articles and
questions into vectors; articles whose meaning is close to a question score high, even when they
share few words.

This notebook needs the **Candle kernel** and the model, fetched once with
`sh demos/candle/fetch-model.sh` (about 90 MB, pinned and hash-checked; Apache-2.0). After that it
runs offline. The articles and questions in `demos/data` were written for this demo: it is a
showcase, not a benchmark of search quality.

A helper: the three articles closest in meaning to a question, best first.

In [1]:
fn top3(articles, docs, encoder, question) {
	let q = encoder.embed([question])?;
	articles.with_dense(candle::similarity(docs, q, ["score"])?)?
		.select_(["id", "title", "score"])?
		.sort(["score"], polars::SortMultipleOptions::new().with_order_descending(true))?
		.head(Some(3))
}

The articles (24 of them, each a few sentences) and the questions.

In [2]:
let articles = polars::read_csv("../data/articles.csv", [("id", "i64"), ("title", "string"), ("text", "string")])?;
let questions = polars::read_csv("../data/questions.csv", [("id", "i64"), ("question", "string")])?;
let asked = questions.strings("question")?;
articles.select_(["id", "title"])?

id,title
i64,string
0,Reset your password
1,Change your email address
2,Turn on two-step sign-in
3,Lost your phone with the authenticator
4,Export notes as PDF
…,…
19,Update your payment card
20,Download invoices
21,Delete your account


Load the model from the fetched files.

In [3]:
let encoder = candle::TextEncoder::load("../models/all-MiniLM-L6-v2")?;
encoder

TextEncoder[bert; 6 layers, 384 dims, max 256 tokens]

Embed every article's text: one 384-number vector per article.

In [4]:
let docs = encoder.embed(articles.strings("text")?)?;
docs

Dense[f32; 24 x 384](e0, e1, e2, e3, e4, e5, e6, e7, …)

Each question, and its three closest articles. Scores are cosine similarities. The model is
small and literal in places: for the lost-phone question it ranks the article on turning on
two-step sign-in just above the one about a lost authenticator, and for the deleted-by-mistake
question it puts "Delete your account" above "Restore a deleted note".

In [5]:
println!("{}", asked[0]);
top3(articles, docs, encoder, asked[0])?

I can't remember my login, how do I get back in?


id,title,score
i64,string,f32
0,Reset your password,0.6512222
2,Turn on two-step sign-in,0.45971993
3,Lost your phone with the authenticator,0.4116352


In [6]:
println!("{}", asked[1]);
top3(articles, docs, encoder, asked[1])?

my phone was stolen and I used it for security codes


id,title,score
i64,string,f32
2,Turn on two-step sign-in,0.4477739
3,Lost your phone with the authenticator,0.40834814
0,Reset your password,0.25700346


In [7]:
println!("{}", asked[2]);
top3(articles, docs, encoder, asked[2])?

how do I get my money back for a payment?


id,title,score
i64,string,f32
18,Request a refund,0.55928826
17,Cancel your subscription,0.3481753
19,Update your payment card,0.3053044


In [8]:
println!("{}", asked[3]);
top3(articles, docs, encoder, asked[3])?

can I keep editing on a plane without internet?


id,title,score
i64,string,f32
9,Work offline,0.4276021
10,Fix sync conflicts,0.37061918
7,Share a notebook with your team,0.28744212


In [9]:
println!("{}", asked[4]);
top3(articles, docs, encoder, asked[4])?

I removed something by mistake yesterday


id,title,score
i64,string,f32
21,Delete your account,0.44901517
6,Restore a deleted note,0.39174065
0,Reset your password,0.24437153


In [10]:
println!("{}", asked[5]);
top3(articles, docs, encoder, asked[5])?

let a coworker edit my notebook


id,title,score
i64,string,f32
7,Share a notebook with your team,0.61989725
9,Work offline,0.3571284
8,Stop sharing a note,0.30632833


The best article for each question.

In [11]:
for question in asked {
	let best = top3(articles, docs, encoder, question)?.column("title")?.str()?.get(0)?.unwrap();
	println!("{} -> {}", question, best);
}

I can't remember my login, how do I get back in? -> Reset your password


my phone was stolen and I used it for security codes -> Turn on two-step sign-in


how do I get my money back for a payment? -> Request a refund


can I keep editing on a plane without internet? -> Work offline


I removed something by mistake yesterday -> Delete your account


let a coworker edit my notebook -> Share a notebook with your team
